# 训练收益标签差异诊断
仅分析已完成训练的1056条成熟样本。检查整体/年度差异、平方差集中度及固定±2日错位，不选取收益最优标签或新参数。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/execution_label_diagnostic_20261015';parent=root/'research/execution_return_training_20261015'
s=json.loads((parent/'summary.json').read_text());manifest=json.loads((parent/'verification_manifest.json').read_text())
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=s['provider_uri'],region='us')
for e in manifest['entries']:
    if e['path'].endswith(('inputs/original_labels','inputs/execution_labels')):
        assert hashlib.sha256((parent/e['path']).read_bytes()).hexdigest()==e['sha256']
with R.uri_context(s['recorder_uri']):
    source=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
    old=source.load_object('inputs/original_labels');new=source.load_object('inputs/execution_labels')
rows=[];lags=[];details={}
for asset in ['soxl','soxx']:
    k=asset+'_forward_open_return'
    z=pd.DataFrame({'old':old[k],'new':new[k]}).loc['2018-10-12':'2022-12-21'].dropna()
    z['delta']=z.new-z.old;details[asset]=z
    for year,g in [('all',z),*list(z.groupby(z.index.year))]:
        sq=g.delta.pow(2); rows.append(dict(asset=asset,year=str(year),n=len(g),correlation=g.old.corr(g.new),mean_delta_bps=1e4*g.delta.mean(),delta_std_bps=1e4*g.delta.std(),sign_disagreement=(g.old*g.new<0).mean(),top10_squared_difference_share=sq.nlargest(10).sum()/sq.sum()))
    for lag in [-2,-1,0,1,2]:lags.append(dict(asset=asset,shift_sessions=lag,correlation=z.old.corr(z.new.shift(lag))))
table=pd.DataFrame(rows);lag_table=pd.DataFrame(lags)
assert all(lag_table[lag_table.asset.eq(a)].sort_values('correlation').iloc[-1].shift_sessions==0 for a in details)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/differences','analysis/metrics','analysis/lags']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
    rec=R.get_recorder();rec.save_objects(artifact_path='analysis',differences=details,metrics=table,lags=lag_table,source_manifest=manifest)
    rec.log_metrics(training_observations=len(details['soxl']),registered_artifact_count=4)
summary={'experiment':card.name,'decision':'diagnostic_only_no_promotion','new_fits':0,'new_backtests':0,'source_recorder':s['recorder_id'],'provider_uri':s['provider_uri'],'region':'us','as_of':s['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':policy,'scope':'Training-only label comparison; no lag selection, no label correction, no new predictive feature. Corporate actions and same-source opening decomposition remain unverified.'}
table.to_csv(card/'label_metrics.csv',index=False);lag_table.to_csv(card/'lag_check.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'source_recorder':s['recorder_id'],'provider_uri':s['provider_uri'],'artifact_policy':policy}))
binding=json.loads((parent/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=card/'qlib_recorder'/str(rec.experiment_id)/rec.id/'artifacts'
(card/'verification_manifest.json').write_text(json.dumps({'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2))
print(table.to_string(index=False));print(lag_table.to_string(index=False))


# 训练标签诊断：无整体日期错位证据，不能用少数异常日解释

本轮0次新训练、0次新回测，复用上一轮Qlib Recorder中2018-10-12至2022-12-21的1056条成熟训练标签，未使用评价期收益选择规则。已执行notebook、保存诊断Recorder并完成NAS逐对象冷恢复哈希验证。

两套标签相关性SOXL为0.8852、SOXX为0.8921；固定前后1/2交易日移动后的相关性绝对值均低于0.05，未发现整体错位一天或两天的证据。年度相关性约0.84至0.92，差异并非只出现于单一年份。

执行收益标签减旧日线标签的均值为SOXL -1.84bp、SOXX -0.25bp，但单日差值标准差分别315.28bp和101.69bp；涨跌符号不一致比例16.29%和16.10%。最大10日平方差占比仅11.31%和8.57%，不能把差异主要归为少数极端日期。上述相关性是两种历史标签的同期一致性，不是预测IC。

结论只支持下一步先做同源价格分解：使用相同分钟来源的09:30开盘、09:47执行价与公司行动调整因子，区分日内开盘区间差异和日线/分钟源差异。尚未完成该分解，不能认定公司行动正确、价格无误或17分钟就是全部原因。禁止删掉最大差异日期、平移标签或挑年份提高回测。

本轮诊断不产生候选策略，不晋升；上一轮执行标签候选失败仍成立，生产模型不变，稳定净SOXX超额仍未取得独立证据。下一项训练必须以通过同源口径核验的标签为前提，不能直接混合旧/新标签或搜索混合权重。
